# Introduction

<center><img src="https://i.imgur.com/9hLRsjZ.jpg" height=400></center>

This dataset was scraped from [nextspaceflight.com](https://nextspaceflight.com/launches/past/?page=1) and includes all the space missions since the beginning of Space Race between the USA and the Soviet Union in 1957!

### Install Package with Country Codes

In [ ]:
# %pip install iso3166

### Upgrade Plotly

Run the cell below if you are working with Google Colab.

In [ ]:
# %pip install --upgrade plotly

Requirement already up-to-date: plotly in /usr/local/lib/python3.6/dist-packages (4.12.0)


### Import Statements

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

# These might be helpful:
from iso3166 import countries
from datetime import datetime, timedelta

### Notebook Presentation

In [3]:
pd.options.display.float_format = '{:,.2f}'.format

### Load the Data

In [4]:
df_data = pd.read_csv('C:\\Users\\Lenovo\\Documents\\github\\Udemy\\Space_Race_Data_Analysis\\mission_launches.csv')

# Preliminary Data Exploration

* What is the shape of `df_data`? 
* How many rows and columns does it have?
* What are the column names?
* Are there any NaN values or duplicates?

In [5]:
print(f"Shape of df_data: {df_data.shape}")
print(f"Number of rows: {df_data.shape[0]:,}")
print(f"Number of columns: {df_data.shape[1]}")

Shape of df_data: (4324, 9)
Number of rows: 4,324
Number of columns: 9


In [6]:
print("Column names:", df_data.columns.tolist())
df_data.head()

Column names: ['Unnamed: 0.1', 'Unnamed: 0', 'Organisation', 'Location', 'Date', 'Detail', 'Rocket_Status', 'Price', 'Mission_Status']


,Unnamed: 0.1,Unnamed: 0,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status
0,0,0,SpaceX,"LC-39A, Kennedy Space Center, Florida, USA","Fri Aug 07, 2020 05:12 UTC",Falcon 9 Block 5 | Starlink V1 L9 & BlackSky,StatusActive,50.0,Success
1,1,1,CASC,"Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...","Thu Aug 06, 2020 04:01 UTC",Long March 2D | Gaofen-9 04 & Q-SAT,StatusActive,29.75,Success
2,2,2,SpaceX,"Pad A, Boca Chica, Texas, USA","Tue Aug 04, 2020 23:57 UTC",Starship Prototype | 150 Meter Hop,StatusActive,NaN,Success
3,3,3,Roscosmos,"Site 200/39, Baikonur Cosmodrome, Kazakhstan","Thu Jul 30, 2020 21:25 UTC",Proton-M/Briz-M | Ekspress-80 & Ekspress-103,StatusActive,65.0,Success
4,4,4,ULA,"SLC-41, Cape Canaveral AFS, Florida, USA","Thu Jul 30, 2020 11:50 UTC",Atlas V 541 | Perseverance,StatusActive,145.0,Success


## Data Cleaning - Check for Missing Values and Duplicates

Consider removing columns containing junk data. 

In [7]:
print("NaN values per column:")
print(df_data.isna().sum())
print(f"\nNumber of duplicated rows: {df_data.duplicated().sum()}")

NaN values per column:
Unnamed: 0.1         0
Unnamed: 0           0
Organisation         0
Location             0
Date                 0
Detail               0
Rocket_Status        0
Price             3360
Mission_Status       0
dtype: int64

Number of duplicated rows: 0


In [8]:
# 'Unnamed: 0.1' and 'Unnamed: 0' are just old index columns saved into the csv -> junk
junk_cols = [col for col in df_data.columns if col.startswith('Unnamed')]
df_data = df_data.drop(columns=junk_cols)

# Price is stored as text (e.g. '5,000.0'), so remove the thousands separator and convert to a number.
# Missing prices stay as NaN.
df_data['Price'] = pd.to_numeric(df_data['Price'].astype(str).str.replace(',', '', regex=False), errors='coerce')

# The index columns made every row look unique - re-check duplicates now that they are gone
print(f"Duplicated rows after dropping junk columns: {df_data.duplicated().sum()}")
df_data = df_data.drop_duplicates().reset_index(drop=True)
print(f"Rows after removing duplicates: {len(df_data):,}")
print(f"Launches with a price: {df_data['Price'].notna().sum():,} | without a price: {df_data['Price'].isna().sum():,}")
df_data.info()

Duplicated rows after dropping junk columns: 1
Rows after removing duplicates: 4,323
Launches with a price: 963 | without a price: 3,360
<class 'pandas.DataFrame'>
RangeIndex: 4323 entries, 0 to 4322
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Organisation    4323 non-null   str    
 1   Location        4323 non-null   str    
 2   Date            4323 non-null   str    
 3   Detail          4323 non-null   str    
 4   Rocket_Status   4323 non-null   str    
 5   Price           963 non-null    float64
 6   Mission_Status  4323 non-null   str    
dtypes: float64(1), str(6)
memory usage: 236.5 KB


## Descriptive Statistics

In [9]:
df_data.describe()

,Price
count,963.00
mean,153.92
std,288.57
min,5.30
25%,40.00
50%,62.00
75%,164.00
max,"5,000.00"


In [10]:
df_data.select_dtypes(exclude='number').describe()

,Organisation,Location,Date,Detail,Rocket_Status,Mission_Status
count,4323,4323,4323,4323,4323,4323
unique,56,137,4319,4278,2,4
top,RVSN USSR,"Site 31/6, Baikonur Cosmodrome, Kazakhstan","Sun Aug 25, 1991 08:40 UTC",Cosmos-3MRB (65MRB) | BOR-5 Shuttle,StatusRetired,Success
freq,1777,235,2,6,3534,3878


# Number of Launches per Company

Create a chart that shows the number of space mission launches by organisation.

In [11]:
launches_by_org = (df_data['Organisation']
                   .value_counts()
                   .rename_axis('Organisation')
                   .reset_index(name='Launches'))
launches_by_org.head(10)

,Organisation,Launches
0,RVSN USSR,1777
1,Arianespace,279
2,General Dynamics,251
3,CASC,250
4,NASA,203
5,VKS RF,201
6,US Air Force,161
7,ULA,140
8,Boeing,136
9,Martin Marietta,114


In [12]:
fig = px.bar(launches_by_org,
             x='Organisation',
             y='Launches',
             color='Launches',
             color_continuous_scale='Viridis',
             title='Number of Space Mission Launches by Organisation')
fig.update_layout(xaxis_tickangle=-60, xaxis_title='Organisation', yaxis_title='Number of Launches')
fig.show()

# Number of Active versus Retired Rockets

How many rockets are active compared to those that are decomissioned? 

In [13]:
rocket_status = (df_data['Rocket_Status']
                 .str.replace('Status', '', regex=False)
                 .value_counts()
                 .rename_axis('Rocket_Status')
                 .reset_index(name='Count'))
rocket_status

,Rocket_Status,Count
0,Retired,3534
1,Active,789


In [14]:
fig = px.pie(rocket_status,
             names='Rocket_Status',
             values='Count',
             hole=0.4,
             title='Active versus Retired Rockets')
fig.update_traces(textinfo='label+value+percent')
fig.show()

# Distribution of Mission Status

How many missions were successful?
How many missions failed?

In [15]:
mission_status = (df_data['Mission_Status']
                  .value_counts()
                  .rename_axis('Mission_Status')
                  .reset_index(name='Count'))

n_success = (df_data['Mission_Status'] == 'Success').sum()
n_not_success = (df_data['Mission_Status'] != 'Success').sum()
print(f"Successful missions: {n_success:,}")
print(f"Failed missions (Failure + Partial Failure + Prelaunch Failure): {n_not_success:,}")
mission_status

Successful missions: 3,878
Failed missions (Failure + Partial Failure + Prelaunch Failure): 445


,Mission_Status,Count
0,Success,3878
1,Failure,339
2,Partial Failure,102
3,Prelaunch Failure,4


In [16]:
fig = px.bar(mission_status,
             x='Mission_Status',
             y='Count',
             color='Mission_Status',
             text_auto=True,
             title='Distribution of Mission Status')
fig.update_layout(showlegend=False, xaxis_title='Mission Status', yaxis_title='Number of Missions')
fig.show()

# How Expensive are the Launches? 

Create a histogram and visualise the distribution. The price column is given in USD millions (careful of missing values). 

In [17]:
price_data = df_data.dropna(subset=['Price'])
print(f"{len(price_data):,} of {len(df_data):,} launches have a price (USD millions)")
price_data['Price'].describe()

963 of 4,323 launches have a price (USD millions)


count     963.00
mean      153.92
std       288.57
min         5.30
25%        40.00
50%        62.00
75%       164.00
max     5,000.00
Name: Price, dtype: float64

In [18]:
fig = px.histogram(price_data,
                   x='Price',
                   nbins=50,
                   marginal='box',
                   title='Distribution of Launch Prices',
                   labels={'Price': 'Launch Price (USD millions)'})
fig.update_layout(yaxis_title='Number of Launches')
fig.show()

# Use a Choropleth Map to Show the Number of Launches by Country

* Create a choropleth map using [the plotly documentation](https://plotly.com/python/choropleth-maps/)
* Experiment with [plotly's available colours](https://plotly.com/python/builtin-colorscales/). I quite like the sequential colour `matter` on this map. 
* You'll need to extract a `country` feature as well as change the country names that no longer exist.

Wrangle the Country Names

You'll need to use a 3 letter country code for each country. You might have to change some country names.

* Russia is the Russian Federation
* New Mexico should be USA
* Yellow Sea refers to China
* Shahrud Missile Test Site should be Iran
* Pacific Missile Range Facility should be USA
* Barents Sea should be Russian Federation
* Gran Canaria should be USA


You can use the iso3166 package to convert the country names to Alpha3 format.

In [19]:
# The country is the last part of the Location string, e.g. "LC-39A, Kennedy Space Center, Florida, USA"
df_data['Country'] = df_data['Location'].str.split(',').str[-1].str.strip()

# Some "countries" are really sites/seas or names that no longer exist -> fix them
country_fixes = {
    'Russia': 'Russian Federation',
    'New Mexico': 'USA',
    'Yellow Sea': 'China',
    'Shahrud Missile Test Site': 'Iran',
    'Pacific Missile Range Facility': 'USA',
    'Barents Sea': 'Russian Federation',
    'Gran Canaria': 'USA',
    'Pacific Ocean': 'Kiribati',   # Sea Launch's Odyssey platform launched from Kiritimati (Kiribati)
}
df_data['Country'] = df_data['Country'].replace(country_fixes)


def get_alpha3(name):
    """Convert a country name to its ISO 3166 alpha-3 code (None if not found)."""
    try:
        return countries.get(name).alpha3
    except KeyError:
        return None


df_data['ISO'] = df_data['Country'].map(get_alpha3)

# Every country should now have a code
print("Countries without an ISO code:", df_data.loc[df_data['ISO'].isna(), 'Country'].unique())
df_data[['Location', 'Country', 'ISO']].head()

Countries without an ISO code: ['Iran' 'North Korea' 'South Korea']


,Location,Country,ISO
0,"LC-39A, Kennedy Space Center, Florida, USA",USA,USA
1,"Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...",China,CHN
2,"Pad A, Boca Chica, Texas, USA",USA,USA
3,"Site 200/39, Baikonur Cosmodrome, Kazakhstan",Kazakhstan,KAZ
4,"SLC-41, Cape Canaveral AFS, Florida, USA",USA,USA


In [20]:
launches_by_country = (df_data.groupby(['Country', 'ISO'])
                       .size()
                       .reset_index(name='Launches'))

fig = px.choropleth(launches_by_country,
                    locations='ISO',
                    color='Launches',
                    hover_name='Country',
                    color_continuous_scale=px.colors.sequential.matter,
                    title='Number of Launches by Country')
fig.update_layout(coloraxis_colorbar=dict(title='Launches'))
fig.show()

# Use a Choropleth Map to Show the Number of Failures by Country


In [21]:
# A failure is any mission that was not a full success (Failure, Partial Failure, Prelaunch Failure)
df_failures = df_data[df_data['Mission_Status'] != 'Success']

failures_by_country = (df_failures.groupby(['Country', 'ISO'])
                       .size()
                       .reset_index(name='Failures'))
failures_by_country.sort_values('Failures', ascending=False).head(10)

,Country,ISO,Failures
11,USA,USA,163
7,Kazakhstan,KAZ,93
10,Russian Federation,RUS,93
2,China,CHN,25
3,France,FRA,18
4,India,IND,13
6,Japan,JPN,13
1,Brazil,BRA,3
0,Australia,AUS,3
8,Kiribati,KIR,3


In [22]:
fig = px.choropleth(failures_by_country,
                    locations='ISO',
                    color='Failures',
                    hover_name='Country',
                    color_continuous_scale=px.colors.sequential.matter,
                    title='Number of Launch Failures by Country')
fig.update_layout(coloraxis_colorbar=dict(title='Failures'))
fig.show()

# Create a Plotly Sunburst Chart of the countries, organisations, and mission status. 

In [23]:
sunburst_data = (df_data.groupby(['Country', 'Organisation', 'Mission_Status'])
                 .size()
                 .reset_index(name='Launches'))
sunburst_data.head()

,Country,Organisation,Mission_Status,Launches
0,Australia,AMBA,Success,1
1,Australia,CECLES,Failure,2
2,Australia,CECLES,Success,1
3,Australia,RAE,Failure,1
4,Australia,RAE,Success,1


In [24]:
fig = px.sunburst(sunburst_data,
                  path=['Country', 'Organisation', 'Mission_Status'],
                  values='Launches',
                  title='Launches by Country, Organisation and Mission Status')
fig.update_layout(width=800, height=800)
fig.show()

# Analyse the Total Amount of Money Spent by Organisation on Space Missions

In [25]:
# Only launches with a known price can be included
money_by_org = (df_data.dropna(subset=['Price'])
                .groupby('Organisation')['Price']
                .sum()
                .sort_values(ascending=False)
                .rename('Total_Spent')
                .reset_index())
money_by_org

,Organisation,Total_Spent
0,NASA,"76,280.00"
1,Arianespace,"16,345.00"
2,ULA,"14,798.00"
3,RVSN USSR,"10,000.00"
4,CASC,"6,310.51"
5,SpaceX,"5,444.00"
6,Northrop,"3,930.00"
7,MHI,"3,532.50"
8,ISRO,"2,177.00"
9,US Air Force,"1,550.92"


In [26]:
fig = px.bar(money_by_org,
             x='Organisation',
             y='Total_Spent',
             color='Total_Spent',
             color_continuous_scale='Viridis',
             title='Total Money Spent by Organisation on Space Missions (USD millions)')
fig.update_layout(xaxis_title='Organisation', yaxis_title='Total Spent (USD millions)')
fig.show()

# Analyse the Amount of Money Spent by Organisation per Launch

In [27]:
price_per_launch = (df_data.dropna(subset=['Price'])
                    .groupby('Organisation')['Price']
                    .agg(['mean', 'count'])
                    .sort_values('mean', ascending=False)
                    .reset_index()
                    .rename(columns={'mean': 'Avg_Price', 'count': 'Priced_Launches'}))
price_per_launch

,Organisation,Avg_Price,Priced_Launches
0,RVSN USSR,"5,000.00",2
1,NASA,511.95,149
2,Boeing,177.29,7
3,Arianespace,170.26,96
4,ULA,151.00,98
5,ILS,101.54,13
6,MHI,95.47,37
7,Martin Marietta,80.16,9
8,US Air Force,59.65,26
9,JAXA,56.00,3


In [28]:
fig = px.bar(price_per_launch,
             x='Organisation',
             y='Avg_Price',
             color='Avg_Price',
             color_continuous_scale='Viridis',
             hover_data=['Priced_Launches'],
             title='Average Amount of Money Spent per Launch by Organisation (USD millions)')
fig.update_layout(xaxis_title='Organisation', yaxis_title='Average Price per Launch (USD millions)')
fig.show()

# Chart the Number of Launches per Year

In [29]:
# Pull the date part out of strings like "Fri Aug 07, 2020 05:12 UTC"
df_data['Launch_Date'] = pd.to_datetime(
    df_data['Date'].str.extract(r'(\w{3} \d{1,2}, \d{4})')[0],
    format='%b %d, %Y')
df_data['Year'] = df_data['Launch_Date'].dt.year
df_data['Month'] = df_data['Launch_Date'].dt.month

launches_per_year = df_data.groupby('Year').size().reset_index(name='Launches')

best_year = launches_per_year.loc[launches_per_year['Launches'].idxmax()]
print(f"Year with the most launches: {int(best_year['Year'])} ({int(best_year['Launches'])} launches)")

Year with the most launches: 1971 (119 launches)


In [30]:
fig = px.bar(launches_per_year,
             x='Year',
             y='Launches',
             title='Number of Launches per Year')
fig.update_layout(xaxis_title='Year', yaxis_title='Number of Launches')
fig.show()

# Chart the Number of Launches Month-on-Month until the Present

Which month has seen the highest number of launches in all time? Superimpose a rolling average on the month on month time series chart. 

In [31]:
# Count launches for every year-month (months with no launches count as 0)
monthly = df_data.groupby(df_data['Launch_Date'].dt.to_period('M')).size()
full_range = pd.period_range(monthly.index.min(), monthly.index.max(), freq='M')
monthly = monthly.reindex(full_range, fill_value=0)

monthly_df = pd.DataFrame({'Month': monthly.index.to_timestamp(), 'Launches': monthly.values})
monthly_df['Rolling_Avg_12M'] = monthly_df['Launches'].rolling(window=12).mean()

peak = monthly_df.loc[monthly_df['Launches'].idxmax()]
print(f"Highest number of launches in a single month: {peak['Month']:%B %Y} ({int(peak['Launches'])} launches)")

Highest number of launches in a single month: December 1971 (18 launches)


In [32]:
fig = px.line(monthly_df,
              x='Month',
              y=['Launches', 'Rolling_Avg_12M'],
              title='Number of Launches Month-on-Month with 12-Month Rolling Average',
              labels={'value': 'Number of Launches', 'Month': 'Date', 'variable': ''})
fig.show()

# Launches per Month: Which months are most popular and least popular for launches?

Some months have better weather than others. Which time of year seems to be best for space missions?

In [33]:
import calendar

launches_per_month = df_data.groupby('Month').size().reset_index(name='Launches')
launches_per_month['Month_Name'] = launches_per_month['Month'].map(lambda m: calendar.month_name[m])

most_popular = launches_per_month.loc[launches_per_month['Launches'].idxmax()]
least_popular = launches_per_month.loc[launches_per_month['Launches'].idxmin()]
print(f"Most popular month: {most_popular['Month_Name']} ({most_popular['Launches']} launches)")
print(f"Least popular month: {least_popular['Month_Name']} ({least_popular['Launches']} launches)")

Most popular month: December (450 launches)
Least popular month: January (268 launches)


In [34]:
fig = px.bar(launches_per_month,
             x='Month_Name',
             y='Launches',
             color='Launches',
             color_continuous_scale='Viridis',
             title='Number of Launches by Month of the Year (all time)')
fig.update_layout(xaxis_title='Month', yaxis_title='Number of Launches')
fig.show()

# How has the Launch Price varied Over Time? 

Create a line chart that shows the average price of rocket launches over time. 

In [35]:
price_by_year = (df_data.dropna(subset=['Price'])
                 .groupby('Year')['Price']
                 .mean()
                 .reset_index(name='Avg_Price'))
price_by_year.head()

,Year,Avg_Price
0,1964,63.23
1,1965,63.23
2,1966,59.00
3,1967,196.62
4,1968,279.20


In [36]:
fig = px.line(price_by_year,
              x='Year',
              y='Avg_Price',
              markers=True,
              title='Average Launch Price over Time (USD millions)')
fig.update_layout(xaxis_title='Year', yaxis_title='Average Launch Price (USD millions)')
fig.show()

# Chart the Number of Launches over Time by the Top 10 Organisations. 

How has the dominance of launches changed over time between the different players? 

In [37]:
top10_orgs = df_data['Organisation'].value_counts().head(10).index.tolist()
print("Top 10 organisations by number of launches:", top10_orgs)

Top 10 organisations by number of launches: ['RVSN USSR', 'Arianespace', 'General Dynamics', 'CASC', 'NASA', 'VKS RF', 'US Air Force', 'ULA', 'Boeing', 'Martin Marietta']


In [38]:
# Launches per year for each of the top 10 organisations (fill years without launches with 0)
org_year_counts = (df_data[df_data['Organisation'].isin(top10_orgs)]
                   .groupby(['Year', 'Organisation'])
                   .size()
                   .unstack(fill_value=0)
                   .stack()
                   .reset_index(name='Launches'))
org_year_counts.head()

,Year,Organisation,Launches
0,1957,Arianespace,0
1,1957,Boeing,0
2,1957,CASC,0
3,1957,General Dynamics,0
4,1957,Martin Marietta,0


In [39]:
fig = px.line(org_year_counts,
              x='Year',
              y='Launches',
              color='Organisation',
              title='Number of Launches over Time by the Top 10 Organisations')
fig.update_layout(xaxis_title='Year', yaxis_title='Number of Launches')
fig.show()

# Cold War Space Race: USA vs USSR

The cold war lasted from the start of the dataset up until 1991. 

In [40]:
# The Soviet Union launched from Russia AND from Kazakhstan (Baikonur), so both count as USSR
cold_war_countries = {
    'USA': 'USA',
    'Russian Federation': 'USSR',
    'Kazakhstan': 'USSR',
}

df_cold = df_data[(df_data['Year'] <= 1991) & (df_data['Country'].isin(list(cold_war_countries)))].copy()
df_cold['Superpower'] = df_cold['Country'].map(cold_war_countries)

In [41]:
print(f"Cold War launches (1957-1991) by USA and USSR: {len(df_cold):,}")
df_cold['Superpower'].value_counts()

Cold War launches (1957-1991) by USA and USSR: 2,432


Superpower
USSR    1770
USA      662
Name: count, dtype: int64

## Create a Plotly Pie Chart comparing the total number of launches of the USSR and the USA

Hint: Remember to include former Soviet Republics like Kazakhstan when analysing the total number of launches. 

In [42]:
cold_war_totals = (df_cold['Superpower']
                   .value_counts()
                   .rename_axis('Superpower')
                   .reset_index(name='Launches'))
cold_war_totals

,Superpower,Launches
0,USSR,1770
1,USA,662


In [43]:
fig = px.pie(cold_war_totals,
             names='Superpower',
             values='Launches',
             title='Total Number of Launches during the Cold War: USA vs USSR')
fig.update_traces(textinfo='label+value+percent')
fig.show()

## Create a Chart that Shows the Total Number of Launches Year-On-Year by the Two Superpowers

In [44]:
launches_yoy = (df_cold.groupby(['Year', 'Superpower'])
                .size()
                .unstack(fill_value=0)
                .stack()
                .reset_index(name='Launches'))
launches_yoy.head()

,Year,Superpower,Launches
0,1957,USA,1
1,1957,USSR,2
2,1958,USA,23
3,1958,USSR,5
4,1959,USA,16


In [45]:
fig = px.line(launches_yoy,
              x='Year',
              y='Launches',
              color='Superpower',
              markers=True,
              title='Number of Launches Year-on-Year: USA vs USSR')
fig.update_layout(xaxis_title='Year', yaxis_title='Number of Launches')
fig.show()

## Chart the Total Number of Mission Failures Year on Year.

In [46]:
# Failures = every mission that was not a full success
failures_yoy = (df_cold[df_cold['Mission_Status'] != 'Success']
                .groupby(['Year', 'Superpower'])
                .size()
                .reset_index(name='Failures'))
failures_yoy.head()

,Year,Superpower,Failures
0,1957,USA,1
1,1958,USA,18
2,1958,USSR,4
3,1959,USA,10
4,1959,USSR,2


In [47]:
fig = px.line(failures_yoy,
              x='Year',
              y='Failures',
              color='Superpower',
              markers=True,
              title='Number of Mission Failures Year-on-Year: USA vs USSR')
fig.update_layout(xaxis_title='Year', yaxis_title='Number of Failures')
fig.show()

## Chart the Percentage of Failures over Time

Did failures go up or down over time? Did the countries get better at minimising risk and improving their chances of success over time? 

In [48]:
cold_summary = (df_cold.groupby(['Year', 'Superpower'])
                .agg(Launches=('Mission_Status', 'size'),
                     Failures=('Mission_Status', lambda s: (s != 'Success').sum()))
                .reset_index())
cold_summary['Failure_Pct'] = cold_summary['Failures'] / cold_summary['Launches'] * 100
cold_summary.head()

,Year,Superpower,Launches,Failures,Failure_Pct
0,1957,USA,1,1,100.00
1,1957,USSR,2,0,0.00
2,1958,USA,23,18,78.26
3,1958,USSR,5,4,80.00
4,1959,USA,16,10,62.50


In [49]:
fig = px.line(cold_summary,
              x='Year',
              y='Failure_Pct',
              color='Superpower',
              markers=True,
              title='Percentage of Failed Launches over Time: USA vs USSR')
fig.update_layout(xaxis_title='Year', yaxis_title='Failure Rate (%)')
fig.show()

In [50]:
# Has the world as a whole got better at launching? Failure rate for ALL launches, every year
world_failure = (df_data.groupby('Year')['Mission_Status']
                 .apply(lambda s: (s != 'Success').mean() * 100)
                 .reset_index(name='Failure_Pct'))
world_failure['Rolling_Avg_5Y'] = world_failure['Failure_Pct'].rolling(window=5).mean()

fig = px.line(world_failure,
              x='Year',
              y=['Failure_Pct', 'Rolling_Avg_5Y'],
              title='Percentage of Failed Launches over Time (all countries)',
              labels={'value': 'Failure Rate (%)', 'variable': ''})
fig.show()

# For Every Year Show which Country was in the Lead in terms of Total Number of Launches up to and including including 2020)

Do the results change if we only look at the number of successful launches? 

In [51]:
df_lead = df_data[df_data['Year'] <= 2020].copy()

# Baikonur (Kazakhstan) was run by the USSR / Russia, so combine them into a single "Russia / USSR"
df_lead['Lead_Country'] = df_lead['Country'].replace({'Russian Federation': 'Russia / USSR',
                                                      'Kazakhstan': 'Russia / USSR'})


def yearly_leader(df):
    """For every year return the country with the most cumulative launches up to and including that year."""
    cumulative = (df.groupby(['Year', 'Lead_Country'])
                  .size()
                  .unstack(fill_value=0)
                  .cumsum())
    return pd.DataFrame({'Leader': cumulative.idxmax(axis=1),
                         'Cumulative_Launches': cumulative.max(axis=1)}).reset_index()


leader_all = yearly_leader(df_lead)
print(leader_all.groupby('Leader')['Year'].agg(['min', 'max', 'count']))

fig = px.bar(leader_all,
             x='Year',
             y='Cumulative_Launches',
             color='Leader',
             title='Country in the Lead by Cumulative Number of Launches (up to and including 2020)')
fig.update_layout(xaxis_title='Year', yaxis_title='Cumulative Launches of the Leader')
fig.show()

                min   max  count
Leader                          
Russia / USSR  1957  2020     54
USA            1958  1967     10


In [52]:
# Same analysis, but only counting successful launches
leader_success = yearly_leader(df_lead[df_lead['Mission_Status'] == 'Success'])

comparison = leader_all.merge(leader_success, on='Year', suffixes=('_all', '_success'))
different_years = comparison[comparison['Leader_all'] != comparison['Leader_success']]
print(f"Years where the leader changes when only successful launches count: {len(different_years)}")
print(different_years[['Year', 'Leader_all', 'Leader_success']])

fig = px.bar(leader_success,
             x='Year',
             y='Cumulative_Launches',
             color='Leader',
             title='Country in the Lead by Cumulative Number of SUCCESSFUL Launches (up to and including 2020)')
fig.update_layout(xaxis_title='Year', yaxis_title='Cumulative Successful Launches of the Leader')
fig.show()

Years where the leader changes when only successful launches count: 1
    Year Leader_all Leader_success
10  1967        USA  Russia / USSR


# Create a Year-on-Year Chart Showing the Organisation Doing the Most Number of Launches

Which organisation was dominant in the 1970s and 1980s? Which organisation was dominant in 2018, 2019 and 2020? 

In [53]:
org_year = (df_data.groupby(['Year', 'Organisation'])
            .size()
            .reset_index(name='Launches'))

# For every year keep the organisation with the most launches
top_org_per_year = org_year.loc[org_year.groupby('Year')['Launches'].idxmax()]
top_org_per_year.tail(10)

,Year,Organisation,Launches
529,2011,ULA,11
541,2012,ULA,10
555,2013,ULA,11
567,2014,ULA,14
569,2015,Arianespace,12
583,2016,CASC,22
611,2017,SpaceX,18
616,2018,CASC,37
632,2019,CASC,27
649,2020,CASC,19


In [54]:
fig = px.bar(top_org_per_year,
             x='Year',
             y='Launches',
             color='Organisation',
             title='Organisation with the Most Launches in Each Year')
fig.update_layout(xaxis_title='Year', yaxis_title='Number of Launches by the Leading Organisation')
fig.show()

In [55]:
def dominant_org(start, end):
    subset = df_data[df_data['Year'].between(start, end)]
    return subset['Organisation'].value_counts().head(3)


print("Most launches 1970-1989:")
print(dominant_org(1970, 1989), "\n")
print("Most launches 2018-2020:")
print(dominant_org(2018, 2020))

Most launches 1970-1989:
Organisation
RVSN USSR           1250
General Dynamics      97
Martin Marietta       84
Name: count, dtype: int64 

Most launches 2018-2020:
Organisation
CASC         83
SpaceX       48
Roscosmos    28
Name: count, dtype: int64
